# Talks markdown generator for academicpages

Takes a TSV of talks with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `talks.py`. Run either from the `markdown_generator` folder after replacing `talks.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases, rather than Stuart's non-standard TSV format and citation style.

In [1]:
import pandas as pd
import os

## Data format

The TSV needs to have the following columns: title, type, url_slug, venue, date, location, talk_url, description, with a header at the top. Many of these fields can be blank, but the columns must be in the TSV.

- Fields that cannot be blank: `title`, `url_slug`, `date`. All else can be blank. `type` defaults to "Talk" 
- `date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. 
    - The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/talks/YYYY-MM-DD-[url_slug]`
    - The combination of `url_slug` and `date` must be unique, as it will be the basis for your filenames

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [2]:
!cat talks.tsv

title	title_short	type	conference	conf_short	conf_type	conf_url	venue	date	location	abstract
"Una forma ""barata"" de garantizar tri�ngulos en una gr�fica (A ""cheap"" way of guarantee triangles in a graph)"	trianglesProb	Contributed	"26th Colloquium �""Victor Neumann-Lara"" of graph theory, combinatorics and its applications"	vnl2011	National		Universidad Aut�noma del Estado de Hidalgo	6.03.11	"Pachuga, Hgo. Mexico"	
Permutaciones: �Funciones biyectivas o simples revolturas? (Permutations: bijections or just jumbles?)	permutation	Contributed	44th National Conference of the Mexican Mathematical Society	smm2011	National		Universidad Aut�noma de San Luis Potos�	12.10.11	"San Luis Potos�, SLP. Mexico"	"Dado un conjunto $A$ con $n$ elementos, las permutaciones de los elementos de dicho conjunto tienen dos grandes formas de estudio:\\ Una de ellas es como objetos de car�cter combinatorio, muy �tiles para resolver variados problemas de conteo relacionados con la idea de revolver los elemento

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [7]:
talks = pd.read_csv("talks.tsv", sep="\t", header=0)
talks

,title,title_short,type,conference,conf_short,conf_type,conf_url,venue,date,location,abstract
0,"Una forma ""barata"" de garantizar triángulos en...",trianglesProb,Contributed,"26th Colloquium ""Victor Neumann-Lara"" of graph...",vnl2011,National,NaN,Universidad Autónoma del Estado de Hidalgo,6.03.11,"Pachuga, Hgo. Mexico",NaN
1,Permutaciones: ¿Funciones biyectivas o simples...,permutation,Contributed,44th National Conference of the Mexican Mathem...,smm2011,National,NaN,Universidad Autónoma de San Luis Potosí,12.10.11,"San Luis Potosí, SLP. Mexico","Dado un conjunto $A$ con $n$ elementos, las pe..."
2,Coloración en gráficas infinitas (Colorings o...,coloringsInfinite,Contributed,Discrete math seminar FisMat UMSNH,dmFisMat,Local,NaN,"Faculty of Physics and Mathematics, UMSNH",15.05.12,"Morelia, Mich. Mexico",NaN
3,Poliedros regulares en pantalla de videojuegos...,regPolyhedra3Torus,Contributed,Discrete math seminar FisMat UMSNH,dmFisMat,Local,NaN,"Faculty of Physics and Mathematics, UMSNH",10.11.12,"Morelia, Mich. Mexico",NaN
4,Poliedros regulares en el $3$-toro (Regular po...,regPolyhedra3Torus,Contributed,"28th Colloquium ""Victor Neumann-Lara"" of graph...",vnl2013,National,NaN,"Palacio Clavijero, Morelia",5.03.13,"Morelia, Mich. Mexico",NaN
5,Poliedros regulares en el $3$-toro (Regular po...,regPolyhedra3Torus,Contributed,46th National Conference of the Mexican Mathem...,smm2013,National,NaN,Universidad Autónoma de Yucatán,28.10.13,"Mérida, Yuc. Mexico",En los 70's Grünbaum publicó una lista de 47 p...
6,Poliedros regulares ¿Aún siguen de moda? (Reg...,regPolyhedra,Contributed,Graduate students seminar CCM,becariosCCM,Local,NaN,"Centro de Ciencias Matemáticas, UNAM Morelia",8.11.13,"Morelia, Mich. Mexico",NaN
7,Poliedros regulares en el $3$-toro (Regular po...,regPolyhedra3Torus,Invitation,Graduate students seminar IM UNAM,becariosIM,Local,NaN,Institute of Mathematics UNAM,1.04.14,"Mexico City, Mexico",NaN
8,Regular polyhedra in the $3$-torus,regPolyhedra3Torus,Contributed,5th Workshop Symmetries in Graphs Maps and Pol...,sigmap2014,International,http://mcs.open.ac.uk/SIGMAP/,ELIM Conference Centre,7.07.14,"West Malvern, U.K.",Since Grünbaum's paper about regular polyhedra...
9,"Simetrías: un paseo por la geometría, los mosa...",mosaicos,Contributed,Degustaciones Matemáticas 2015,degustaciones2015,Local,NaN,"Faculty of Physics and Mathematics, UMSNH",14.02.15,"Morelia, Mich. Mexico",NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [4]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;",
    "?": "&quest;"
    "¿": "&iquest;"
    "á": "&aacute;",
    "ä": "&auml;",
    "é": "&eacute;",
    "í": "&iacute;",
    "ó": "&oacute;",
    "ú": "&uacute;",
    "Á": "&Aacute;",
    "É": "&Eacute;",
    "Í": "&Iacute;",
    "Ó": "&Oacute;",
    "Ú": "&Uacute;",
    "ñ": "&ntilde;",
    "Ñ": "&Ntilde;",
    }

def html_escape(text):
    if type(text) is str:
        return "".join(html_escape_table.get(c,c) for c in text)
    else:
        return "False"

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [5]:
loc_dict = {}

for row, item in talks.iterrows():
    
    md_filename = str(item.date) + "-" + item.url_slug + ".md"
    html_filename = str(item.date) + "-" + item.url_slug 
    year = item.date[:4]
    
    md = "---\ntitle: \""   + item.title + '"\n'
    md += "collection: talks" + "\n"
    
    if len(str(item.type)) > 3:
        md += 'type: "' + item.type + '"\n'
    else:
        md += 'type: "Talk"\n'
    
    md += "permalink: /talks/" + html_filename + "\n"
    
    if len(str(item.venue)) > 3:
        md += 'venue: "' + item.venue + '"\n'
        
    if len(str(item.location)) > 3:
        md += "date: " + str(item.date) + "\n"
    
    if len(str(item.location)) > 3:
        md += 'location: "' + str(item.location) + '"\n'
           
    md += "---\n"
    
    
    if len(str(item.talk_url)) > 3:
        md += "\n[More information here](" + item.talk_url + ")\n" 
        
    
    if len(str(item.description)) > 3:
        md += "\n" + html_escape(item.description) + "\n"
        
        
    md_filename = os.path.basename(md_filename)
    #print(md)
    
    with open("../_talks/" + md_filename, 'w') as f:
        f.write(md)

These files are in the talks directory, one directory below where we're working from.

In [6]:
!ls ../_talks

2012-03-01-talk-1.md	  2014-02-01-talk-2.md
2013-03-01-tutorial-1.md  2014-03-01-talk-3.md


In [7]:
!cat ../_talks/2013-03-01-tutorial-1.md

---
title: "Tutorial 1 on Relevant Topic in Your Field"
collection: talks
type: "Tutorial"
permalink: /talks/2013-03-01-tutorial-1
venue: "UC-Berkeley Institute for Testing Science"
date: 2013-03-01
location: "Berkeley CA, USA"
---

[More information here](http://exampleurl.com)

This is a description of your tutorial, note the different field in type. This is a markdown files that can be all markdown-ified like any other post. Yay markdown!
